In [ ]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
# from shapely.geometry import Polygon 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import contextily as ctx
from adjustText import adjust_text
import matplotlib
import csv

from matplotlib import cm
from matplotlib.colors import Normalize
import matplotlib as mpl

from rasterstats import zonal_stats

import rioxarray as rxr
import matplotlib.colors as mpl_cl

from sklearn.preprocessing import MinMaxScaler
import matplotlib.patches as mpatches
import contextily as ctx

import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from sklearn.preprocessing import MinMaxScaler

import matplotlib.lines as mlines

# Geostatistical libraries
import pysal
from pysal.lib import weights
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib_scalebar.scalebar import ScaleBar

import xarray as xr 

from libpysal import weights

## Read the Ghana Shapefile Data & Filter for Accra 

In [ ]:
main_shapefile = gpd.read_file('/Users/kwamedonkor/Downloads/UW/Research/Ghana Statistical Services Data /1. 2021 Population and Housing Census/Shapefiles for all 16 regions/Districts_261.shp')

### Read PQR data from the public dashboard (using 2021 data - has most sites represented) 

In [ ]:
df_saidi_2021 = pd.read_csv('/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/nLine_download_dashboard/space_grouping_public_dataset/SAIDI (hours) - Yearly of 2024-01-01-data-2025-04-03 12_22_57.csv')

### Extract geometry column from saidi data 

In [ ]:
# Function to extract the geometry data from the original df (convert the dictionary to geometry) 
def convert_to_geometry(boundary):
    boundary_dict = json.loads(boundary)
    if 'type' in boundary_dict and boundary_dict['type'] == 'Feature':
        return shape(boundary_dict['geometry'])
    else:
        raise ValueError("Invalid boundary data")


# Convert the 'space_boundary' column to geometry
df_saidi_2021['geometry'] = df_saidi_2021['space_boundary'].apply(convert_to_geometry)

# Convert the df to a gdf of the nLine sites 
gdf_sites_2021 = gpd.GeoDataFrame(df_saidi_2021, geometry='geometry')

### Find centroid & reproject (nLine sites) 

In [ ]:
# set CRS as EPSG:4326 (lat & lon coordinates)
gdf_sites_2021 = gdf_sites_2021.set_crs('EPSG:4326')

gdf_sites_2021 = gdf_sites_2021.drop(columns=['space_boundary', 'est_uncertainty', 'avg_sites_reporting'])

## Reproject first 
gdf_sites_2021 = gdf_sites_2021.to_crs(epsg=32630)

# find the centroid of each site 
gdf_sites_2021['geometry'] = gdf_sites_2021['geometry'].centroid

In [ ]:
# rename gdf 
new_sites = gdf_sites_2021.copy()

### save nline sites gdf for use 
# new_sites.to_file("/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/filtered_nLine_sites_2021_revised.shp")

#### Shift the sites that are out of place ('out at sea')

In [ ]:

## function to perform shifting 

def translate_geometries(gdf, space_grouping_name, x_offset, y_offset):

    # Apply translation to the filtered subset of geometries
    gdf.loc[gdf['space_grouping'] == space_grouping_name, 'geometry'] = (
        gdf.loc[gdf['space_grouping'] == space_grouping_name, 'geometry'].translate(xoff=x_offset, yoff=y_offset)
    )
    return gdf


### site '57'
new_sites = translate_geometries(new_sites, 57, 0, +500)

### site '54'
new_sites = translate_geometries(new_sites, 54, 0, +200)

### site '116'
new_sites = translate_geometries(new_sites, 116, -100, +400)

## Read the Study Area EA shapefile 

In [ ]:
# read the Greater Accra Enumeration Areas shapefile 
ea_shapefile = gpd.read_file('/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Enumeration Areas/Archive/GAMA_20200923_eaBorder.shp')

# reproject 
ea_shapefile = ea_shapefile.to_crs('EPSG:32630')

### Filter the PQR sites in the Study Area 

In [ ]:
district_gdf = ea_shapefile.dissolve(by='DIST_CODE')
district_gdf = district_gdf.reset_index()

# find sites in the study area  
intersecting_sites = gpd.sjoin(
    new_sites, district_gdf, 
    how="inner", 
    predicate='intersects' 
)

### The list of all PQR sites in study area 

In [ ]:
intersecting_site_list = np.sort(intersecting_sites['space_grouping'].unique())

df_site_ids = pd.DataFrame(intersecting_site_list, columns=['site_id'])

### save master list of PQR sites 
# df_site_ids.to_csv('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/df_site_ids_geo_analysis.csv')

## Workflow for CESI Creation 

In [ ]:
# renaming gdf as new_ea 
new_ea = ea_shapefile.copy()

#### Slum Index per EA 

In [ ]:
# read slum index per EA data 
slum_index = pd.read_csv('/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/final_probabilities.csv')

# select only the needed columns 
slum_index = slum_index[['ea_code9ch', 'probabilities']]
slum_index = slum_index.rename(columns = {'probabilities':'slum_index'})


### Intersection: All EAs and Slum Index 

In [ ]:
# convert to integer 
new_ea['ea_code9ch'] = new_ea['ea_code9ch'].astype(int)

# merge EA gdf with slum index data 
new_ea_r1 = pd.merge(new_ea, slum_index, on = 'ea_code9ch', how = 'outer')

new_ea_r1 = new_ea_r1[['ea_code9ch','geometry', 'slum_index']]

#### Elevation & Slope 

In [ ]:
#### Read Slope raster file & run a zonal stats to find mean slope per EA 
saved_slope_fp = "/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/slope_gh.tif"
stats_slope = zonal_stats(new_ea, saved_slope_fp, stats=["mean"], raster_out=False)
# Convert the result data to df & rename columns 
slope_df = pd.DataFrame(stats_slope).rename(columns={"mean":"mean_slope"})
# Add to new_ea df 
new_ea_r1 = new_ea_r1.join(slope_df)


#### Do same for the mean Elevation per EA 
saved_dem_fp = "/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/dem_gh.tif"
stats_dem = zonal_stats(new_ea, saved_dem_fp, stats=["mean"], raster_out=False)
# Convert to Df & rename columns 
dem_df = pd.DataFrame(stats_dem).rename(columns={"mean":"mean_elev"})
# Add to new_ea df 
new_ea_r1 = new_ea_r1.join(dem_df)

In [ ]:
#### Normalize & Invert (so that higher elevation and slope values have lower flood risk) 

# List of columns to transform
cols = ["mean_elev", "mean_slope"]  

# Apply a log10 normalization because of huge disparity in min and max values 
d = np.log10(new_ea_r1[cols])

# Normalize each column & invert 
log_df = 1 - (d - d.min()) / (d.max() - d.min())

## rename columns 
for col in log_df.columns:
    log_df[col + '_log_inverted'] = log_df[col]

log_df = log_df.drop(columns=cols)

## Add the log_inverted columns back to the df 
new_ea_r1 = new_ea_r1.join(log_df)

### NDVI (based on the satellite coverage, we needed 2 scenes to get the whole Greater Accra) 

#### *(Scene for East side)* 

In [ ]:
# specify portion of the EAs as in the East category using their centroids 
east_filtered_new_ea = new_ea[new_ea.geometry.apply(
    lambda geom: geom.centroid.x > 805000 if geom.geom_type in ['Polygon', 'MultiPolygon'] 
    else geom.x > 805000)].reset_index(drop = True)

In [ ]:
## read east side raster file and run a zonal stats (mean NDVI)
saved_ndvi_east_fp = "/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/ndvi_corrected_gh_east.tif"

stats_ndvi_east = zonal_stats(east_filtered_new_ea, saved_ndvi_east_fp, stats=["mean"], raster_out=False)
ndvi_df_east = pd.DataFrame(stats_ndvi_east).rename(columns={"mean":"mean_ndvi_east"})

east_filtered_new_ea = east_filtered_new_ea[['ea_code9ch', 'geometry']]
east_filtered_new_ea = east_filtered_new_ea.join(ndvi_df_east)

#### *(Scene for West side)* 

In [ ]:
# the rest of the EAs are therefore in the West category 
rest_of_eas = new_ea[ ~( new_ea['ea_code9ch'].isin(east_filtered_new_ea['ea_code9ch'].unique()) ) ].reset_index(drop = True)

In [ ]:
# similar to East -> read West file and run zonal stats (mean NDVI)
saved_ndvi_west_fp = "/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/ndvi_corrected_gh_west.tif"

stats_ndvi_west = zonal_stats(rest_of_eas, saved_ndvi_west_fp, stats=["mean"], raster_out=False)
ndvi_df_west = pd.DataFrame(stats_ndvi_west).rename(columns={"mean":"mean_ndvi_west"})

rest_of_eas = rest_of_eas[['ea_code9ch', 'geometry']]
rest_of_eas = rest_of_eas.join(ndvi_df_west)

#### *(Combining both West & East NDVIs)* 

In [ ]:
# concatenate both East & West EAs into one file 
rest_of_eas_new = rest_of_eas.rename(columns = {'mean_ndvi_west':'mean_ndvi'})
east_filtered_new_ea_rev1 = east_filtered_new_ea.rename(columns = {'mean_ndvi_east':'mean_ndvi'})

ndvi_total = pd.concat([rest_of_eas_new, east_filtered_new_ea_rev1]).reset_index(drop = True).drop(columns = ['geometry'])

#### Merge NDVI data with new_ea Gdf

In [ ]:
new_ea_r1 = pd.merge(new_ea_r1, ndvi_total, on = 'ea_code9ch')   
  
# normalize the NDVI values from 0 to 1 and invert (so that higher NDVI values/greenery have lower flood risk)
scaler = MinMaxScaler()
new_ea_r1['normalized_mean_ndvi'] = scaler.fit_transform(new_ea_r1[['mean_ndvi']])
new_ea_r1['inverted_normalized_ndvi'] = 1 - new_ea_r1['normalized_mean_ndvi']

### Total Rainfall Exposure 

In [ ]:
# read in total rainfall exposure metrics for each EA 
rainfall_df_new = pd.read_csv('/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/Geo_Interpol_Metrics_All_New/total_rainfall_5019_EAs.csv').drop(columns = ['Unnamed: 0'])

# normalize from 0 to 1 
scaler = MinMaxScaler()
rainfall_df_new['normalized_rainfall'] = scaler.fit_transform(rainfall_df_new[['Total_Rainfall']])

## merge tihs also on ea_code9ch 
new_ea_r1 = pd.merge(new_ea_r1, rainfall_df_new, on = 'ea_code9ch', how = 'outer')

### Air Quality  

In [ ]:
# read pm2.5 raster file & run a zonal stats too (mean 2022 annaul aq per EA)
saved_2022_aq_data_fp = "/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/aq_22_float.tif"
stats_2022_aq = zonal_stats(new_ea, saved_2022_aq_data_fp, stats=["mean"], raster_out=False, all_touched=True)

# Convert the result to df & rename columns 
aq_22_df = pd.DataFrame(stats_2022_aq).rename(columns={"mean":"pm2.5_22"})

In [ ]:
# add AQ data to the main df 
new_ea_r1 = new_ea_r1.join(aq_22_df)

# normalize from 0 to 1 
scaler = MinMaxScaler()
new_ea_r1['norm_pm2.5'] = scaler.fit_transform(new_ea_r1[['pm2.5_22']])

#### Keep only relevant columns in the combined df 

In [ ]:
new_ea_r1 = new_ea_r1.drop(columns = ['mean_slope', 'mean_elev', 'mean_ndvi', 'normalized_mean_ndvi', 'pm2.5_22'])

### Slope, elevation, rainfall & NDVI into Flood Risk (additive aggregation)

In [ ]:
cols_flood_risk = [
    'mean_elev_log_inverted', 
    'mean_slope_log_inverted', 
    'normalized_rainfall', 
    'inverted_normalized_ndvi'
]

# Flood risk (equal weighting) 
new_ea_r1['flood_risk_add'] = new_ea_r1[cols_flood_risk].mean(axis=1, skipna=True)

### Climate Exposure & Sensitivity Index (CESI) -- combining Flood Risk, Slum Index and PM2.5 

In [ ]:
cols_static_score = [
    'flood_risk_add', 
    'slum_index', 
    'norm_pm2.5'
]

# Climate Vulnerability 
new_ea_r1['climate_vul_add'] = new_ea_r1[cols_static_score].mean(axis=1, skipna=True)

### Climate Vulnerability Index Plot 

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10), dpi=300)

# Plot all EAs
new_ea_r1.plot(ax=ax, color='grey', alpha=0.6, edgecolor='black', lw=0.7)

# Plot rainfall exposure and store colorbar
cax = new_ea_r1.plot(
    ax=ax,
    column='climate_vul_add',
    cmap='inferno_r',
    edgecolor='black',
    lw=0.7,
    legend=True,
    legend_kwds={'label': 'Climate Vulnerability Index', 'shrink': 0.8, 'pad': 0.03}
)

# Access and update the colorbar label
cbar = cax.get_figure().axes[-1]  # This gets the colorbar axis
cbar.set_ylabel('Climate Vulnerability Index', weight='bold', labelpad=20, fontsize=12)

# Title and formatting
ax.set_title('Climate Vulnerability Index', fontsize=16, pad=15)
ax.set_axis_off()

# Basemap
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=new_ea_r1.crs)

# "No Data" legend
no_data_patch = mpatches.Patch(color='grey', alpha=0.5, label="No Data")
ax.legend(handles=[no_data_patch], loc='lower right', fontsize=10, handleheight=1.5, handlelength=2)


# plt.savefig('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/Images/CVI_spatial.png', bbox_inches='tight', dpi=300)
# path_1 = '/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/Slide_Deck/New_Slide_Deck_figures/cvi_all.png'
# plt.savefig(path_1, bbox_inches='tight', dpi=300)

plt.show()

### Saving: CESI-EA mapping data 

In [ ]:
cvi_ea_mapping = new_ea_r1.copy()[['ea_code9ch', 'climate_vul_add']]
# cvi_ea_mapping.to_csv('/Users/kwamedonkor/Downloads/UW/Research/Quals/Output_Files/cvi_ea_mapping.csv')

### Saving: EA names & codes  

In [ ]:
global_cesi_names = new_ea.copy()[['ea_code9ch', 'LOC_NAME', 'BASE_NAM']]
# global_cesi_names.to_csv('/Users/kwamedonkor/Library/CloudStorage/OneDrive-UW/Quals/Output_Files/global_cesi_names.csv')